# DE_IP_2024 — домашнее задание №2

## Исследование методов подавления шума

**Задание:** зашумить изображение гауссовым, постоянным и импульсным шумом; протестировать медианный, гауссов, билатеральный фильтр и фильтр нелокальных средних с различными параметрами; определить лучший результат фильтрации.

В этой работе постоянный шум реализован как аддитивный положительный шум: к каждому пикселю прибавляется постоянная величина `40`. После сложения значения ограничиваются диапазоном `0...255`.

In [ ]:
from pathlib import Path
import urllib.request

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['image.cmap'] = 'gray'

IMAGE_PATH = Path('sar_1.jpg')
IMAGE_URL = (
    'https://sc04.alicdn.com/kf/'
    'Aa9b902d1d96a42428a9fd49e593430272.jpg'
)

if not IMAGE_PATH.exists():
    print('Файл sar_1.jpg не найден. Загружаю изображение...')
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image_bgr = cv2.imread(str(IMAGE_PATH))
if image_bgr is None:
    raise FileNotFoundError(f'Не удалось прочитать изображение: {IMAGE_PATH.resolve()}')

image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
height, width = image_gray.shape
print(f'Размер изображения: {width} x {height}')

plt.figure(figsize=(8, 6))
plt.imshow(image_gray)
plt.title('Исходное изображение sar_1.jpg')
plt.axis('off');

## 1. Формирование тестовых шумов

Используется фиксированный генератор случайных чисел, поэтому результаты воспроизводимы. Гауссов шум имеет нулевое среднее и стандартное отклонение `35`. Импульсный шум заменяет случайные пиксели на чёрные и белые с общей долей `8%`. Постоянный шум одинаково увеличивает яркость всех пикселей на `40`.

In [ ]:
rng = np.random.default_rng(42)

def add_gaussian_noise(image, sigma=35, generator=None):
    if generator is None:
        generator = np.random.default_rng(42)
    noise = generator.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_constant_noise(image, value=40):
    noisy = image.astype(np.int16) + int(value)
    return np.clip(noisy, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(image, amount=0.08, salt_ratio=0.5, generator=None):
    if generator is None:
        generator = np.random.default_rng(42)
    noisy = image.copy()
    count = int(amount * image.size)
    rows = generator.integers(0, image.shape[0], count)
    cols = generator.integers(0, image.shape[1], count)
    salt_count = int(count * salt_ratio)
    noisy[rows[:salt_count], cols[:salt_count]] = 255
    noisy[rows[salt_count:], cols[salt_count:]] = 0
    return noisy

image_gaussian = add_gaussian_noise(image_gray, sigma=35, generator=rng)
image_constant = add_constant_noise(image_gray, value=40)
image_salt_pepper = add_salt_pepper_noise(image_gray, amount=0.08, generator=rng)

noisy_images = [
    ('Исходное изображение', image_gray),
    ('Гауссов шум, sigma = 35', image_gaussian),
    ('Постоянный шум, value = +40', image_constant),
    ('Импульсный шум, amount = 8%', image_salt_pepper),
]

plt.figure(figsize=(16, 8))
for index, (title, image) in enumerate(noisy_images, start=1):
    plt.subplot(2, 2, index)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
plt.tight_layout();

## 2. Метрики качества

MSE показывает среднюю квадратичную ошибку. PSNR выражается в децибелах и увеличивается при улучшении изображения. SSIM оценивает сохранение структуры изображения. Чем меньше MSE и чем больше PSNR/SSIM, тем лучше результат.

In [ ]:
def mse(reference, image):
    reference = reference.astype(np.float32)
    image = image.astype(np.float32)
    return float(np.mean((reference - image) ** 2))

def psnr(reference, image):
    error = mse(reference, image)
    if error == 0:
        return float('inf')
    return float(10 * np.log10((255.0 ** 2) / error))

def ssim_score(reference, image):
    reference = reference.astype(np.float32)
    image = image.astype(np.float32)
    kernel_size = (11, 11)
    sigma = 1.5
    mu_ref = cv2.GaussianBlur(reference, kernel_size, sigma)
    mu_img = cv2.GaussianBlur(image, kernel_size, sigma)
    mu_ref_sq = mu_ref * mu_ref
    mu_img_sq = mu_img * mu_img
    mu_product = mu_ref * mu_img
    sigma_ref_sq = cv2.GaussianBlur(reference * reference, kernel_size, sigma) - mu_ref_sq
    sigma_img_sq = cv2.GaussianBlur(image * image, kernel_size, sigma) - mu_img_sq
    sigma_product = cv2.GaussianBlur(reference * image, kernel_size, sigma) - mu_product
    c1 = (0.01 * 255) ** 2
    c2 = (0.03 * 255) ** 2
    numerator = (2 * mu_product + c1) * (2 * sigma_product + c2)
    denominator = (mu_ref_sq + mu_img_sq + c1) * (sigma_ref_sq + sigma_img_sq + c2)
    return float(np.mean(numerator / (denominator + 1e-12)))

def metrics(reference, image):
    return {
        'MSE': mse(reference, image),
        'PSNR': psnr(reference, image),
        'SSIM': ssim_score(reference, image),
    }

for noise_name, noisy_image in [
    ('Гауссов шум', image_gaussian),
    ('Постоянный шум', image_constant),
    ('Импульсный шум', image_salt_pepper),
]:
    print(noise_name + ':', metrics(image_gray, noisy_image))

## 3. Набор фильтров и параметров

In [ ]:
filter_specs = [
    ('Медианный 3x3', lambda image: cv2.medianBlur(image, 3)),
    ('Медианный 5x5', lambda image: cv2.medianBlur(image, 5)),
    ('Медианный 7x7', lambda image: cv2.medianBlur(image, 7)),
    ('Гауссов 3x3, sigma=0', lambda image: cv2.GaussianBlur(image, (3, 3), 0)),
    ('Гауссов 5x5, sigma=1', lambda image: cv2.GaussianBlur(image, (5, 5), 1)),
    ('Гауссов 7x7, sigma=2', lambda image: cv2.GaussianBlur(image, (7, 7), 2)),
    ('Билатеральный d=5', lambda image: cv2.bilateralFilter(image, 5, 25, 25)),
    ('Билатеральный d=9', lambda image: cv2.bilateralFilter(image, 9, 50, 50)),
    ('Билатеральный d=13', lambda image: cv2.bilateralFilter(image, 13, 75, 75)),
    ('NLM h=5', lambda image: cv2.fastNlMeansDenoising(image, None, 5, 7, 21)),
    ('NLM h=10', lambda image: cv2.fastNlMeansDenoising(image, None, 10, 7, 21)),
    ('NLM h=20', lambda image: cv2.fastNlMeansDenoising(image, None, 20, 7, 21)),
]
print(f'Количество вариантов фильтрации: {len(filter_specs)}')

In [ ]:
def run_experiment(noisy_image, noise_name):
    rows = []
    results = {}
    for filter_name, filter_function in filter_specs:
        filtered = filter_function(noisy_image)
        results[filter_name] = filtered
        row = {'Шум': noise_name, 'Фильтр': filter_name}
        row.update(metrics(image_gray, filtered))
        rows.append(row)
    return rows, results

experiments = {
    'Гауссов': image_gaussian,
    'Постоянный': image_constant,
    'Импульсный': image_salt_pepper,
}

experiment_rows = {}
experiment_results = {}
for noise_name, noisy_image in experiments.items():
    experiment_rows[noise_name], experiment_results[noise_name] = run_experiment(
        noisy_image, noise_name
    )

def sort_key(row):
    return (-row['SSIM'], -row['PSNR'], row['MSE'])

best_results = {
    noise_name: sorted(rows, key=sort_key)[0]
    for noise_name, rows in experiment_rows.items()
}

for noise_name, best in best_results.items():
    print(f"{noise_name}: {best['Фильтр']}, SSIM={best['SSIM']:.4f}")

## 4. Сравнение результатов

Для каждого вида шума результаты сортируются по SSIM, затем по PSNR. В таблице первая строка является лучшей по выбранному критерию.

In [ ]:
def print_table(rows, title):
    print(title)
    print(f"{'Фильтр':<28} {'MSE':>12} {'PSNR':>10} {'SSIM':>10}")
    print('-' * 64)
    for row in sorted(rows, key=sort_key):
        print(f"{row['Фильтр']:<28} {row['MSE']:>12.2f} {row['PSNR']:>10.2f} {row['SSIM']:>10.4f}")

for noise_name, rows in experiment_rows.items():
    print_table(rows, noise_name + ' шум')
    print()

In [ ]:
def show_top_results(noisy_image, result_dict, rows, title, count=5):
    selected_rows = sorted(rows, key=sort_key)[:count]
    plt.figure(figsize=(18, 10))
    plt.subplot(2, 3, 1)
    plt.imshow(noisy_image)
    plt.title('Зашумлённое изображение')
    plt.axis('off')
    for index, row in enumerate(selected_rows, start=2):
        plt.subplot(2, 3, index)
        plt.imshow(result_dict[row['Фильтр']])
        plt.title(
            f"{row['Фильтр']}\n"
            f"PSNR={row['PSNR']:.2f}, SSIM={row['SSIM']:.4f}"
        )
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout();

for noise_name, noisy_image in experiments.items():
    show_top_results(
        noisy_image, experiment_results[noise_name], experiment_rows[noise_name],
        'Пять лучших результатов: ' + noise_name + ' шум'
    )

## 5. Итоговый вывод

Постоянный шум не создаёт случайных выбросов, а систематически увеличивает яркость пикселей. Поэтому обычные сглаживающие фильтры могут не полностью устранить такой сдвиг яркости: они уменьшают локальные колебания, но не выполняют коррекцию смещения. Для постоянного шума в дополнение к фильтрации можно использовать компенсацию известного значения — вычесть `40` и снова ограничить результат диапазоном `0...255`. В итоговом сравнении ниже автоматически указывается лучший фильтр по SSIM.

In [ ]:
constant_compensated = np.clip(
    image_constant.astype(np.int16) - 40, 0, 255
).astype(np.uint8)
constant_compensation_metrics = metrics(image_gray, constant_compensated)

print('ИТОГ')
for noise_name, best in best_results.items():
    print(f"{noise_name} шум: лучший фильтр — {best['Фильтр']}")
    print(f"  MSE={best['MSE']:.2f}, PSNR={best['PSNR']:.2f}, SSIM={best['SSIM']:.4f}")
print('Постоянный шум после компенсации известного смещения:')
print(
    f"  MSE={constant_compensation_metrics['MSE']:.2f}, "
    f"PSNR={constant_compensation_metrics['PSNR']:.2f}, "
    f"SSIM={constant_compensation_metrics['SSIM']:.4f}"
)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_constant)
plt.title('Постоянный шум +40')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(constant_compensated)
plt.title('После компенсации -40')
plt.axis('off')
plt.tight_layout();